In [1]:
import pandas as pd
import json

In [19]:
import tqdm

In [126]:
import math
import numpy as np

In [51]:
yelp_review = pd.read_json('/Users/lujiewen/Desktop/papers/datasets/yelp_json/yelp_academic_dataset_review.json', lines=True)

In [52]:
trailed_yelp_review = yelp_review[['user_id', 'business_id', 'stars', 'date']]

In [53]:
trailed_yelp_review['date'] = pd.to_datetime(trailed_yelp_review['date'])
trailed_yelp_review.sort_values(by='date', ascending=True, inplace=True)
trailed_yelp_review.reset_index(drop=True, inplace=True)

/tmp/ipykernel_81277/3792270611.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trailed_yelp_review['date'] = pd.to_datetime(trailed_yelp_review['date'])
/tmp/ipykernel_81277/3792270611.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trailed_yelp_review.sort_values(by='date', ascending=True, inplace=True)


In [58]:
user_ids = trailed_yelp_review.user_id.unique()
user_review_count = trailed_yelp_review.groupby('user_id')['stars'].count().reset_index()
user_review_count.columns = ['user_id', 'review_count']
candidate_user_ids = set(user_review_count[(user_review_count['review_count'] >= 20) & (user_review_count['review_count'] <= 50)]['user_id'].values)

In [61]:
sampled_yelp_review = trailed_yelp_review[trailed_yelp_review['user_id'].map(lambda x: x in candidate_user_ids)]
sampled_yelp_review.reset_index(drop=True, inplace=True)

In [62]:
data_len = len(sampled_yelp_review)
start_time, split_time, end_time = sampled_yelp_review['date'][0], sampled_yelp_review['date'][
            round(0.8 * data_len)], sampled_yelp_review['date'][data_len - 1]

In [63]:
start_time

Timestamp('2005-03-08 05:05:58')

In [64]:
split_time

Timestamp('2019-08-05 21:34:50')

In [65]:
end_time

Timestamp('2022-01-19 19:44:03')

In [67]:
len(set(sampled_yelp_review['user_id']))

32628

In [68]:
32628 / 540847

0.060327597268728494

### User first action time

In [75]:
user_first_action_time = sampled_yelp_review.groupby('user_id')['date'].min().reset_index()

In [76]:
user_warm_list = list(set(user_first_action_time[user_first_action_time['date'] <= split_time]['user_id']))
user_cold_list = list(set(user_first_action_time[user_first_action_time['date'] > split_time]['user_id']))

In [78]:
len(user_warm_list)

31514

In [79]:
len(user_cold_list)

1114

In [133]:
set(user_warm_list) & set(user_cold_list)

set()

### Item first action time

In [139]:
item_first_action_time = sampled_yelp_review.groupby('business_id')['date'].min().reset_index()
item_warm_list = list(set(item_first_action_time[item_first_action_time['date'] <= split_time]['business_id']))
item_cold_list = list(set(item_first_action_time[item_first_action_time['date'] > split_time]['business_id']))

In [141]:
len(item_warm_list)

102787

In [142]:
len(item_cold_list)

13562

#### Business table

In [80]:
business = pd.read_json('/home/deepheart/Downloads/yelp_dataset/yelp_academic_dataset_business.json', lines=True)

In [82]:
trailed_business = business[['business_id', 'postal_code', 'stars']]

In [84]:
candidate_business = set(sampled_yelp_review['business_id'])
sampled_business = trailed_business[trailed_business['business_id'].map(lambda x: x in candidate_business)]
sampled_business.reset_index(drop=True, inplace=True)

In [85]:
sampled_business

,business_id,postal_code,stars
0,tUFrWirKiKi_TAnsVWINQQ,85711,3.5
1,MTSW4McQd7CbVtyjqoe9mw,19107,4.0
2,mWMc6_wTdE0EUBKIGXDVfA,18054,4.5
3,n_0UpQx1hsNbnPUSlodU8w,63144,2.5
4,k0hlBqXX-Bt0vf1op7Jr1w,63123,3.0
...,...,...,...
116344,hn9Toz3s-Ei3uZPt7esExA,T5T 1K8,4.5
116345,IUQopTMmYQG-qRtBk-8QnA,T6J 5H2,3.0
116346,c8GjPIOTGVmIemT7j5_SyQ,37204,4.0
116347,mtGm22y5c2UHNXDFAjaPNw,62025,4.0


### User Table

In [86]:
user_table = pd.read_json('/home/deepheart/Downloads/yelp_dataset/yelp_academic_dataset_user.json', lines=True)

In [88]:
user_table.columns

Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='object')

In [89]:
trailed_user_table = user_table[['user_id', 'friends', 'fans', 'average_stars']]

In [92]:
sampled_user_table = trailed_user_table[trailed_user_table['user_id'].map(lambda x: x in candidate_user_ids)]
sampled_user_table.reset_index(drop=True, inplace=True)

### Create User2Id mapping and Business2ID mapping

In [96]:
user2id = {uid: i for i, uid in enumerate(candidate_user_ids)}
item2id = {item: i for i, item in enumerate(candidate_business)}

### Convert to training data

In [97]:
rating_data = sampled_yelp_review.copy()
rating_data.columns = ['user', 'item', 'rating', 'time'] 
rating_data['user'] = rating_data['user'].map(lambda x: user2id[x])
rating_data['item'] = rating_data['item'].map(lambda x: item2id[x])

In [163]:
output_dir = '/home/deepheart/Desktop/PAML/yelp/'

In [164]:
rating_data.to_csv(output_dir + 'rating.dat', header=False, sep='\t')

In [99]:
user_fans_data = sampled_user_table[['user_id', 'fans']].copy()
user_fans_data.columns = ['user', 'fans']
user_fans_data['user'] = user_fans_data['user'].map(lambda x: user2id[x])

In [100]:
user_fans_data.head()

,user,fans
0,31547,28
1,31721,0
2,32548,75
3,2184,316
4,14089,2


In [165]:
user_fans_data.to_csv(output_dir + 'user_fans.dat', header=False, sep='\t')

In [101]:
user_avg_rating_data = sampled_user_table[['user_id', 'average_stars']].copy()
user_avg_rating_data.columns = ['user', 'avgrating']
user_avg_rating_data['user'] = user_avg_rating_data['user'].map(lambda x: user2id[x])

In [102]:
user_avg_rating_data

,user,avgrating
0,31547,4.27
1,31721,3.98
2,32548,3.41
3,2184,3.61
4,14089,3.87
...,...,...
32623,2859,4.55
32624,2804,3.03
32625,4738,2.68
32626,27722,3.76


In [166]:
user_avg_rating_data.to_csv(output_dir + 'user_avgrating.dat', header=False, sep='\t')

In [120]:
def process_friends_list(x):
    friends = x.split(",")
    friends_ids = []
    for friend in friends:
        friend = friend.strip()
        if friend not in user2id:
            continue
        friends_ids.append(str(user2id[friend]))
    return " ".join(friends_ids)

In [121]:
user_friends_data = sampled_user_table[['user_id', 'friends']].copy()
user_friends_data.columns = ['user', 'friends']
user_friends_data['user'] = user_friends_data['user'].map(lambda x: user2id[x])
user_friends_data['friends'] = user_friends_data['friends'].map(process_friends_list)

In [122]:
user_friends_data.head()

,user,friends
0,31547,15734 3488 15455 17568
1,31721,9102 17761
2,32548,23566 16051 1965 5833 4318 3074 16819 14972 11...
3,2184,20012 16717 168 24109 5691 4381 28044 32137 28...
4,14089,


In [167]:
user_avg_rating_data.to_csv(output_dir + 'user_friends.dat', header=False, sep='\t')

In [103]:
item_stars_data = sampled_business[['business_id', 'stars']]
item_stars_data.columns = ['item', 'stars']
item_stars_data['item'] = item_stars_data['item'].map(lambda x: item2id[x])

/tmp/ipykernel_81277/3461435018.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_stars_data['item'] = item_stars_data['item'].map(lambda x: item2id[x])


In [105]:
item_stars_data.head()

,item,stars
0,29569,3.5
1,65071,4.0
2,80497,4.5
3,7922,2.5
4,19904,3.0


In [168]:
item_stars_data.to_csv(output_dir + 'item_stars.dat', header=False, sep='\t')

In [106]:
item_postalcode_data = sampled_business[['business_id', 'postal_code']].copy()
item_postalcode_data.columns = ['item', 'postalcode']
item_postalcode_data['item'] = item_postalcode_data['item'].map(lambda x: item2id[x])

In [107]:
item_postalcode_data.head()

,item,postalcode
0,29569,85711
1,65071,19107
2,80497,18054
3,7922,63144
4,19904,63123


In [169]:
item_postalcode_data.to_csv(output_dir + 'item_postalcode.dat', header=False, sep='\t')

## Generate Training Json data

### 1. meta training & warmup & item_cold test

* the data is from the user before split time

In [124]:
rating_data.head()

,user,item,rating,time
0,20451,79332,5,2005-03-08 05:05:58
1,20451,68592,5,2005-03-08 05:10:34
2,20451,70383,4,2005-03-08 05:16:12
3,20451,46400,4,2005-03-09 03:03:03
4,20451,87326,1,2005-03-09 03:13:41


In [146]:
item_warm_set = set([item2id[x] for x in item_warm_list])
item_cold_set = set([item2id[x] for x in item_cold_list])

In [147]:
len(item_cold_set)

13562

In [148]:
len(item_warm_set)

102787

In [151]:
meta_training = dict()
meta_training_y = dict()
warmup_training = dict()
warmup_training_y = dict()
item_cold_testing = dict()
item_cold_testing_y = dict()
for user_id in tqdm.tqdm(user_warm_list):
    uid = user2id[user_id]
    user_ratings = rating_data[rating_data['user'] == uid]
    train_items, cold_test_items = [], []
    train_ratings, cold_test_labels = [], []
    for item, label in zip(user_ratings['item'], user_ratings['rating']):
        ## add cold item to item_cold_testing
        if item in item_cold_set:
            cold_test_items.append(item)
            cold_test_labels.append(label)
        else:
            train_items.append(item)
            train_ratings.append(label)
    if train_items:
        meta_training[str(uid)] = train_items
        meta_training_y[str(uid)] = train_ratings
    if cold_test_items:
        item_cold_testing[str(uid)] = cold_test_items
        item_cold_testing_y[str(uid)] = cold_test_labels
    ## sample 10% of items for warm-up
    if len(train_items) > 10:
        sampled_items = math.floor(len(train_items) * 0.1)
        sampled_idx = np.random.choice(len(train_items), sampled_items, replace=False)
        sampled_items = []
        sampled_labels = []
        for s_id in sampled_idx:
            sampled_items.append(train_items[s_id])
            sampled_labels.append(train_ratings[s_id])
        warmup_training[str(uid)] = sampled_items
        warmup_training_y[str(uid)] = sampled_labels

100%|███████████████████████████████████| 31514/31514 [00:26<00:00, 1182.80it/s]


In [170]:
with open(output_dir + 'warm_up.json', 'w') as f:
    json.dump(warmup_training, f)
with open(output_dir + 'warm_up_y.json', 'w') as f:
    json.dump(warmup_training_y, f)
with open(output_dir + 'meta_training.json', 'w') as f:
    json.dump(warmup_training, f)
with open(output_dir + 'meta_training_y.json', 'w') as f:
    json.dump(meta_training_y, f)
with open(output_dir + 'item_cold_testing.json', 'w') as f:
    json.dump(item_cold_testing, f)
with open(output_dir + 'item_cold_testing_y.json', 'w') as f:
    json.dump(item_cold_testing_y, f)

In [158]:
meta_training['4']

[99824,
 22623,
 91317,
 65543,
 9336,
 1272,
 89044,
 40011,
 100356,
 41599,
 106285,
 61448,
 43160,
 40794,
 99119,
 55230,
 72269,
 11403,
 91950]

In [157]:
warmup_training['4']

[40011]

In [156]:
item_cold_testing['4']

[77333]

### 2. user cold testing

In [136]:
user_cold_testing = dict()
user_cold_etsting_y = dict()
for user_id in tqdm.tqdm(user_cold_list):
    uid = user2id[user_id]
    user_ratings = rating_data[rating_data['user'] == uid]
    items = []
    ratings = []
    for item, label in zip(user_ratings['item'], user_ratings['rating']):
        items.append(item)
        ratings.append(label)
    user_cold_testing[str(uid)] = items
    user_cold_etsting_y[str(uid)] = ratings

100%|█████████████████████████████████████| 1114/1114 [00:00<00:00, 1348.33it/s]


In [162]:
len(user_cold_testing)

1114

In [171]:
with open(output_dir + 'user_cold_testing.json', 'w') as f:
    json.dump(user_cold_testing, f)
with open(output_dir + 'user_cold_testing_y.json', 'w') as f:
    json.dump(user_cold_etsting_y, f)

### 3. user & item cold testing

In [159]:
user_item_cold_testing = dict()
user_item_cold_etsting_y = dict()
for user_id in tqdm.tqdm(user_cold_list):
    uid = user2id[user_id]
    user_ratings = rating_data[rating_data['user'] == uid]
    items = []
    ratings = []
    for item, label in zip(user_ratings['item'], user_ratings['rating']):
        if item not in item_cold_set:
            continue
        items.append(item)
        ratings.append(label)
    if items:
        user_item_cold_testing[str(uid)] = items
        user_item_cold_etsting_y[str(uid)] = ratings

100%|█████████████████████████████████████| 1114/1114 [00:00<00:00, 1279.59it/s]


In [161]:
len(user_item_cold_testing)

1087

In [172]:
with open(output_dir + 'user_and_item_cold_testing.json', 'w') as f:
    json.dump(user_item_cold_testing, f)
with open(output_dir + 'user_and_item_cold_testing_y.json', 'w') as f:
    json.dump(user_item_cold_etsting_y, f)